<a href="https://colab.research.google.com/github/prishaa09/solarflarephase2/blob/main/Solar_Flare_Phase_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ================== SETUP (run this cell) ==================
# Stable pins for SunPy 5.1; install optional deps used by sunpy.net
!pip -q install "numpy==1.26.4" "pandas==2.2.2" "astropy==7.1.0" "sunpy==5.1.0" zeep drms

from pathlib import Path
import pandas as pd
from sunpy.net import Fido, attrs as a
from sunpy import timeseries as ts
from google.colab import files

# --------------- CONFIG ---------------
START = "2015-01-01 00:00"    # change as needed
END   = "2015-03-31 23:59"    # change as needed
OUT_DIR = Path("/content/data_goes")
CSV_NAME = f"goes_xrs_{START[:10]}_to_{END[:10]}_1min.csv"
CADENCE = "1min"
# --------------------------------------


def fetch_goes_xrs(start: str, end: str):
    """
    Download GOES XRS data and return a single TimeSeries.
    Handles multiple daily files by merging them safely.
    """
    res = Fido.search(a.Time(start, end), a.Instrument("XRS"))
    files = Fido.fetch(res)

    # May return a single TimeSeries or a list of TimeSeries
    ts_list = ts.TimeSeries(files)

    if isinstance(ts_list, list):
        # Merge all chunks via DataFrame, sort, and drop duplicate times
        combined_df = pd.concat([t.to_dataframe() for t in ts_list])
        combined_df = combined_df.sort_index()
        combined_df = combined_df[~combined_df.index.duplicated(keep="last")]
        return ts.GenericTimeSeries(combined_df)
    else:
        return ts_list

def clean_goes(ts_obj, cadence="1min") -> pd.DataFrame:
    """
    TimeSeries -> DataFrame
    Normalize index to seconds, enforce uniqueness, standardize columns,
    clean, resample to 1-min, add flux ratio.
    """
    df = ts_obj.to_dataframe().copy()

    # --- 1) Normalize time index to tz-naive, whole seconds ---
    # (always tz-aware first, then drop tz; round to 1 second)
    idx = pd.to_datetime(df.index, errors="coerce", utc=True).tz_convert(None).round("S")
    df.index = idx
    # drop bad timestamps, sort
    df = df[~df.index.isna()].sort_index()

    # --- 2) ENFORCE UNIQUE TIMESTAMPS (critical fix) ---
    # keep the last sample if multiple rows share the same timestamp
    df = df[~df.index.duplicated(keep="last")]

    # --- 3) Standardize column names across GOES generations ---
    rename = {}
    for c in df.columns:
        lc = c.lower()
        if ("1-8" in lc) or ("xrsb" in lc) or ("long" in lc): rename[c] = "flux_1_8A"
        if ("0.5-4" in lc) or ("xrsa" in lc) or ("short" in lc): rename[c] = "flux_0.5_4A"
        if ("quality" in lc) or ("qc_flag" in lc): rename[c] = "quality_flag"
        if "sat" in lc: rename[c] = "satellite_id"
    if rename:
        df = df.rename(columns=rename)

    # --- 4) Basic cleaning ---
    for c in ("flux_1_8A", "flux_0.5_4A"):
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
            df.loc[df[c] < 0, c] = None

    if "quality_flag" in df.columns:
        df = df[df["quality_flag"].isna() | (df["quality_flag"] == 0)]

    # (defensive) ensure uniqueness again before resample
    if df.index.has_duplicates:
        df = df[~df.index.duplicated(keep="last")]

    # --- 5) Resample to uniform cadence ---
    df = df.resample(cadence).mean()

    # --- 6) Derived feature & tidy ---
    if {"flux_1_8A", "flux_0.5_4A"}.issubset(df.columns):
        df["flux_ratio"] = df["flux_0.5_4A"] / df["flux_1_8A"]

    df = df.dropna(subset=[c for c in ["flux_1_8A", "flux_0.5_4A"] if c in df.columns], how="all")
    return df.reset_index().rename(columns={"index": "time_utc"})


def save_csv(df: pd.DataFrame, out_dir: Path, name: str) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / name
    df.to_csv(path, index=False)
    print(f"✅ Saved → {path}")
    return path


# ------------------ RUN ------------------
print(f"Fetching GOES XRS {START} → {END}")
ts_obj = fetch_goes_xrs(START, END)
# df = clean_goes(ts_obj, CADENCE)
df = ts_obj.to_dataframe().copy()
csv_path = save_csv(df, OUT_DIR, CSV_NAME)

print("Shape:", df.shape)
print(df.head())

# Download the CSV to your computer
files.download(str(csv_path))

Fetching GOES XRS 2015-01-01 00:00 → 2015-03-31 23:59


Files Downloaded:   0%|          | 0/332 [00:00<?, ?file/s]

✅ Saved → /content/data_goes/goes_xrs_2015-01-01_to_2015-03-31_1min.csv
Shape: (6967865, 4)
                                 xrsa      xrsb  xrsa_quality  xrsb_quality
2015-01-01 00:00:00.000  2.566501e-08  0.000001             0             0
2015-01-01 00:00:01.321  2.366749e-08  0.000001             0             0
2015-01-01 00:00:03.368  2.630058e-08  0.000001             0             0
2015-01-01 00:00:05.415  2.498404e-08  0.000001             0             0
2015-01-01 00:00:07.465  2.366749e-08  0.000001             0             0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>